<a href="https://colab.research.google.com/github/Swayam-dot-cmd/neural-information-retrieval/blob/main/notebooks/04_hyde_improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q beir sentence-transformers transformers scikit-learn

In [ ]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

out_dir = "./datasets"
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

doc_ids = list(corpus.keys())
corpus_texts = [corpus[doc_id]["text"] for doc_id in doc_ids]

print("Corpus:", len(corpus_texts))
print("Queries:", len(queries))

In [ ]:
dense_model = SentenceTransformer('BAAI/bge-small-en')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
generator_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("Generator loaded")

In [ ]:
def generate_hypo_doc(query):
    prompt = f"""
    Question: {query}

    Write a precise, factual, and scientific paragraph that answers this question.
    Use correct terminology and avoid vague statements.
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = generator_model.generate(
        **inputs,
        max_length=128
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
corpus_embeddings = dense_model.encode(
    corpus_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Corpus encoded")

In [ ]:
def hyde_retrieve(query, top_k=10):
    hypo_doc = generate_hypo_doc(query)

    query_emb = dense_model.encode([query], convert_to_numpy=True)[0]
    hypo_emb = dense_model.encode([hypo_doc], convert_to_numpy=True)[0]

    # weighted combination
    final_emb = 0.7 * query_emb + 0.3 * hypo_emb

    scores = cosine_similarity([final_emb], corpus_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    return [doc_ids[i] for i in top_indices]

In [ ]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)

    return len(retrieved_k & relevant_set) / len(relevant_set)

In [ ]:
recalls = []

for qid in list(queries.keys())[:50]:
    query = queries[qid]

    retrieved_docs = hyde_retrieve(query, top_k=10)
    relevant_docs = list(qrels[qid].keys())

    if len(relevant_docs) == 0:
        continue

    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

print("Improved HyDE (50):", sum(recalls)/len(recalls))

In [ ]:
recalls = []

for qid in queries:
    query = queries[qid]

    retrieved_docs = hyde_retrieve(query, top_k=10)
    relevant_docs = list(qrels[qid].keys())

    if len(relevant_docs) == 0:
        continue

    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

avg_recall = sum(recalls) / len(recalls)

print("Improved HyDE Recall@10:", avg_recall)